# M5 Forecasting — ETL Pipeline: **Load Phase**

**Project:** Retail-Demand-Forecasting
**Stage:** `03_load` — connect, load, and validate against PostgreSQL
**Author:** Data Engineering / Analytics Team

---

## Purpose of this notebook

This notebook implements the **discovery and validation** work for the Load phase of the M5
Forecasting ETL pipeline. It connects to an already-running PostgreSQL instance via
**SQLAlchemy**, loads the star-schema tables produced by the Transform phase
(`data/processed/*.csv`), and validates that the load was correct — row counts, primary keys,
foreign keys, and referential integrity.

Concretely, the notebook:

1. Connects to PostgreSQL through SQLAlchemy and verifies the connection.
2. Verifies the required tables exist (creating them from an explicit, inline DDL script if
   they don't — see the scope note below).
3. Explains the loading order dictated by foreign key dependencies.
4. Loads `calendar`, `products`, `stores`, `prices`, `sales` in that order, using batch
   inserts via `pandas.DataFrame.to_sql(..., method="multi", chunksize=...)`.
5. Validates row counts, primary keys, foreign keys, and referential integrity with SQL.
6. Measures and discusses load performance and scalability.

> ⚠️ **Scope boundary — read this first**
>
> This notebook is **exploratory and diagnostic**, not a production pipeline:
> - No reusable functions or modules are built — every step is written out inline so it can be
>   inspected, timed, and validated independently.
> - The full production ETL pipeline (retries, incremental/upsert loads, orchestration,
>   scheduling, monitoring) is explicitly **out of scope** — this notebook only establishes
>   *that a correct one-shot load is possible* and documents what a production implementation
>   would need to do differently (see the final "Load Phase Decisions" section).
> - Table creation here uses a plain inline `CREATE TABLE IF NOT EXISTS` script for
>   demonstration purposes. In production this belongs in a proper migration tool
>   (e.g. Alembic) — not in an ETL notebook.


## 1. Imports and Connection Configuration

We use `sqlalchemy` for the database connection/engine, `pandas` for batch loading via
`to_sql()`, `pathlib` to locate the processed data directory portably, and `time` to measure
load duration.

**Credentials are read from environment variables, not hardcoded.** This is a small but
important production-mindedness detail even in an exploratory notebook: connection strings
containing passwords should never be committed to a notebook. Sensible localhost defaults are
provided as fallbacks purely so this notebook is runnable out of the box in a local dev
environment.


In [ ]:
from pathlib import Path
import os
import time
import warnings

import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text
from sqlalchemy.exc import IntegrityError

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

# Connection parameters: environment variables first, local-dev defaults as fallback.
# In production these would come from a secrets manager, never from notebook source.
PG_HOST = os.environ.get("PGHOST", "localhost")
PG_PORT = os.environ.get("PGPORT", "5432")
PG_DATABASE = os.environ.get("PGDATABASE", "m5_forecasting")
PG_USER = os.environ.get("PGUSER", "m5_user")
PG_PASSWORD = os.environ.get("PGPASSWORD", "m5_password")

CONNECTION_URL = sa.URL.create(
    drivername="postgresql+psycopg2",
    username=PG_USER,
    password=PG_PASSWORD,
    host=PG_HOST,
    port=int(PG_PORT),
    database=PG_DATABASE,
)

print(f"Target host     : {PG_HOST}:{PG_PORT}")
print(f"Target database : {PG_DATABASE}")
print(f"Target user      : {PG_USER}")
print("Password         : ******** (loaded from environment, not printed)")


**What was done:** Imported `sqlalchemy`, `pandas`, `pathlib`, and `time`, then built a
`sqlalchemy.URL` from environment variables (with local-dev fallbacks) instead of a raw,
hardcoded connection string.

**Why it matters:** Using `sqlalchemy.URL.create()` instead of an f-string avoids credential
leakage through string formatting mistakes and correctly escapes special characters that can
appear in passwords. Sourcing values from `os.environ` means this exact cell works unchanged
across a laptop, CI, and a production scheduler — only the environment differs, not the code.

**Load Phase step:** Configuration — establishes *how* we will connect, before actually
connecting.


## 2. Locate the Processed Data Directory

As in the Extract and Transform notebooks, we resolve the project root by walking upward from
the current working directory until a marker directory is found — here, `data/processed/`,
which is expected to contain the star-schema outputs of the Transform phase:
`calendar.csv`, `products.csv`, `stores.csv`, `prices.csv`, `sales.csv`.


In [ ]:
current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / "data" / "processed").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate a 'data/processed' directory above {current}. "
        "Run this notebook from inside the Retail-Demand-Forecasting project, "
        "after the Transform phase has produced the star-schema CSVs."
    )

data_processed_dir = project_root / "data" / "processed"
print(f"Project root         : {project_root}")
print(f"Processed data dir   : {data_processed_dir}")

expected_files = ["calendar.csv", "products.csv", "stores.csv", "prices.csv", "sales.csv"]
for fname in expected_files:
    fpath = data_processed_dir / fname
    status = "OK" if fpath.exists() else "MISSING"
    size_kb = fpath.stat().st_size / 1024 if fpath.exists() else float("nan")
    print(f"  {fname:<16} [{status}]  {size_kb:,.1f} KB" if fpath.exists() else f"  {fname:<16} [{status}]")


**What was done:** Located `data/processed/` with the same marker-based `pathlib`
search used throughout the pipeline, and confirmed all five expected star-schema CSVs are
present before attempting to load any of them.

**Why it matters:** Failing fast here — before opening a database connection — means a missing
Transform-phase output is reported clearly, rather than surfacing as a confusing empty-table
load later.

**Load Phase step:** Source verification — confirms *what* we are about to load exists and is
where we expect it.


## 3. Connect to PostgreSQL via SQLAlchemy and Verify the Connection

We create a SQLAlchemy `Engine` (a connection pool/factory, not a live connection by itself),
then explicitly open a connection and run a trivial query (`SELECT 1`) to prove the database is
actually reachable and authenticating correctly — rather than assuming `create_engine()`
succeeding means the database is up (it doesn't; `create_engine` is lazy).


In [ ]:
engine = sa.create_engine(CONNECTION_URL, pool_pre_ping=True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1")).scalar()
    print(f"Connection test (SELECT 1) returned: {result}")

    db_version = conn.execute(text("SELECT version()")).scalar()
    print(f"\nPostgreSQL version:\n{db_version}")

    current_db = conn.execute(text("SELECT current_database()")).scalar()
    current_user = conn.execute(text("SELECT current_user")).scalar()
    print(f"\nConnected to database '{current_db}' as user '{current_user}'")

print("\nConnection verified successfully.")


**What was done:** Created a SQLAlchemy `Engine` with `pool_pre_ping=True` (so stale
connections are detected and refreshed automatically), then opened a real connection and ran
`SELECT 1`, `SELECT version()`, and `SELECT current_database()/current_user` to positively
confirm connectivity, server version, and that we authenticated as the expected user against
the expected database.

**Why it matters:** `create_engine()` alone never touches the network — it only validates the
URL. The *only* reliable way to know the database is actually reachable is to execute a query
against it. Checking `current_database()`/`current_user` also guards against the easy mistake
of accidentally connecting to the wrong environment (e.g. a shared dev database instead of the
intended one).

**Load Phase step:** Connection verification (Task 2) — a hard prerequisite before Task 3
(verifying tables) or any load can proceed.


## 4. Verify That the Required Tables Exist

We use SQLAlchemy's `inspect()` to list the tables actually present in the target schema and
compare against the five tables this notebook needs: `calendar`, `products`, `stores`,
`prices`, `sales`.

If any are missing, we create them from an explicit DDL script with primary keys and foreign
keys declared up front — so PostgreSQL itself enforces referential integrity during loading,
rather than us checking for violations only after the fact. (In a production setup this DDL
would live in a versioned migration, not inline in a notebook — see the scope note at the top.)


In [ ]:
inspector = sa.inspect(engine)
existing_tables = set(inspector.get_table_names())
required_tables = {"calendar", "products", "stores", "prices", "sales"}

print(f"Existing tables in database: {sorted(existing_tables) if existing_tables else '(none)'}")
print(f"Required tables            : {sorted(required_tables)}")
print(f"Missing tables              : {sorted(required_tables - existing_tables) or '(none)'}")


**What was done:** Used `sqlalchemy.inspect(engine).get_table_names()` to list every
table currently in the target database, then compared it against the five tables this load
requires.

**Why it matters:** This is Task 3 exactly as specified — verifying table existence *before*
attempting to load, so a missing table produces a clear, early signal instead of a
`ProgrammingError` raised mid-way through a batch insert.

**Load Phase step:** Target verification (Task 3).


In [ ]:
DDL_STATEMENTS = {
    "calendar": '''
        CREATE TABLE IF NOT EXISTS calendar (
            d               TEXT PRIMARY KEY,
            date            DATE NOT NULL,
            wm_yr_wk        INTEGER NOT NULL,
            weekday         TEXT,
            wday            INTEGER,
            month           INTEGER,
            year            INTEGER,
            event_name_1    TEXT,
            event_type_1    TEXT,
            event_name_2    TEXT,
            event_type_2    TEXT,
            snap_ca         SMALLINT,
            snap_tx         SMALLINT,
            snap_wi         SMALLINT
        );
    ''',
    "products": '''
        CREATE TABLE IF NOT EXISTS products (
            item_id  TEXT PRIMARY KEY,
            dept_id  TEXT NOT NULL,
            cat_id   TEXT NOT NULL
        );
    ''',
    "stores": '''
        CREATE TABLE IF NOT EXISTS stores (
            store_id  TEXT PRIMARY KEY,
            state_id  TEXT NOT NULL
        );
    ''',
    "prices": '''
        CREATE TABLE IF NOT EXISTS prices (
            store_id     TEXT NOT NULL REFERENCES stores(store_id),
            item_id      TEXT NOT NULL REFERENCES products(item_id),
            wm_yr_wk     INTEGER NOT NULL,
            sell_price   NUMERIC(10, 2) NOT NULL,
            PRIMARY KEY (store_id, item_id, wm_yr_wk)
        );
    ''',
    "sales": '''
        CREATE TABLE IF NOT EXISTS sales (
            item_id   TEXT NOT NULL REFERENCES products(item_id),
            store_id  TEXT NOT NULL REFERENCES stores(store_id),
            d         TEXT NOT NULL REFERENCES calendar(d),
            sales     INTEGER NOT NULL,
            PRIMARY KEY (item_id, store_id, d)
        );
    ''',
}

# Create in dependency order: dimensions first, then tables that reference them.
creation_order = ["calendar", "products", "stores", "prices", "sales"]

with engine.begin() as conn:
    for table_name in creation_order:
        conn.execute(text(DDL_STATEMENTS[table_name]))
        print(f"Ensured table exists: {table_name}")

# Re-inspect to confirm.
inspector = sa.inspect(engine)
existing_tables = set(inspector.get_table_names())
print(f"\nTables now present: {sorted(existing_tables)}")
assert required_tables.issubset(existing_tables), "Not all required tables exist after DDL run."
print("All required tables confirmed present.")


**What was done:** Defined the DDL for all five tables inline (primary keys on
`calendar.d`, `products.item_id`, `stores.store_id`, a composite key on `prices`, and a
composite key on `sales`, with foreign keys wired to their parent dimension tables), executed
it with `CREATE TABLE IF NOT EXISTS` so the script is safe to re-run, then re-inspected the
database to confirm all five tables exist.

**Why it matters:** Declaring primary and foreign keys **at table-creation time** means
PostgreSQL enforces them automatically during every subsequent `INSERT` — invalid data is
rejected by the database itself, which is a stronger and more reliable guarantee than checking
for problems only after loading. Note that `prices.wm_yr_wk` is **not** declared as a foreign
key to `calendar` — `calendar`'s primary key is the daily grain `d`, and `wm_yr_wk` is not
unique per row in `calendar` (many days share one week), so no valid FK target exists for it
there without first building a separate `week` dimension table. That's flagged as a known
design gap in the final Decisions section rather than silently ignored.

**Load Phase step:** Schema preparation, supporting Task 3 (table existence) and providing the
foundation for Tasks 8–9 (PK/FK validation) later in this notebook.


## 5. Loading Order: Why It's Dictated by Foreign Key Dependencies

PostgreSQL enforces foreign key constraints **at insert time**: a row cannot be inserted into a
child table if the parent row it references doesn't exist yet. This means the load order is
not a style choice — it's forced by the dependency graph declared in the DDL above:

```
calendar   (no dependencies)
products   (no dependencies)
stores     (no dependencies)
    │
    ├──> prices   (references products.item_id, stores.store_id)
    │
    └──> sales    (references products.item_id, stores.store_id, calendar.d)
```

**Required order:** `calendar` → `products` → `stores` → `prices` → `sales`
(`calendar`/`products`/`stores` can technically load in any order relative to each other, since
none of them depend on one another — but all three **must** complete before `prices` or
`sales` can begin, and `prices`/`sales` themselves have no dependency on each other, only on
the three dimension tables above them).

Loading `sales` before `calendar`/`products`/`stores` exist, for example, would cause
PostgreSQL to reject every row with a foreign key violation — this is the database actively
protecting referential integrity, which is exactly the outcome we want during development.


## 6. Loading the Data (Batch Inserts, in Dependency Order)

For each table we: read the processed CSV with `pandas`, load it with
`DataFrame.to_sql(..., if_exists="append", index=False, method="multi", chunksize=...)`, and
time the operation. `method="multi"` batches multiple rows into a single `INSERT` statement
(rather than one round-trip per row), and `chunksize` caps how many rows go into each batch —
both are the main levers for efficient batch loading with plain `pandas`/`SQLAlchemy`.

We load the tables **one cell per table**, in the exact dependency order established above, so
each load's success can be confirmed before the next (dependent) one begins.


In [ ]:
load_timings = {}  # populated as we go: table_name -> seconds
load_row_counts_source = {}  # table_name -> rows read from CSV


**What was done:** Initialized two plain dictionaries to accumulate timing and
source-row-count results across the following load cells (not a function — just shared state
for this notebook's own bookkeeping, consistent with the "no reusable modules" scope).

**Load Phase step:** Setup for Tasks 6, 7, and 11 (batch loading, row-count verification, and
timing measurement).


In [ ]:
# --- Load: calendar ---
calendar_df = pd.read_csv(data_processed_dir / "calendar.csv")

# PostgreSQL folds unquoted identifiers to lowercase; rename the SNAP columns to match
# the lowercase column names declared in the DDL above, so pandas' auto-generated
# (and correctly-quoted) INSERT statements target the columns that actually exist.
calendar_df = calendar_df.rename(columns={
    "snap_CA": "snap_ca",
    "snap_TX": "snap_tx",
    "snap_WI": "snap_wi",
})

load_row_counts_source["calendar"] = len(calendar_df)

start = time.perf_counter()
calendar_df.to_sql(
    "calendar",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)
elapsed = time.perf_counter() - start
load_timings["calendar"] = elapsed

print(f"Loaded 'calendar': {len(calendar_df):,} rows in {elapsed:.3f}s "
      f"({len(calendar_df) / elapsed:,.0f} rows/sec)" if elapsed > 0 else f"Loaded 'calendar': {len(calendar_df):,} rows")


**What was done:** Loaded `calendar.csv` (the first table with no dependencies) into
the `calendar` table using batched `to_sql()`, and recorded row count and elapsed time. Before
loading, the mixed-case `snap_CA`/`snap_TX`/`snap_WI` source columns were renamed to lowercase
to match the table's actual column names.

**Why it matters:** `calendar` has no foreign key dependencies, so it's always safe to load
first — this establishes the "no dependency" starting point of the load order explained above.
The rename step matters because PostgreSQL folds **unquoted** identifiers (like the ones in our
DDL) to lowercase, while `pandas.to_sql()` generates a **quoted**, case-preserving `INSERT` for
every DataFrame column — so a raw-CSV column named `snap_CA` would otherwise target a
column literally called `"snap_CA"`, which doesn't exist (the table only has `snap_ca`). This
is a common, easy-to-miss mismatch between CSV-sourced column names and SQL identifier folding
rules, worth catching explicitly rather than debugging via a cryptic `UndefinedColumn` error.

**Load Phase step:** Task 5 (load `calendar`), Task 6 (batch loading), Task 11 (timing).


In [ ]:
# --- Load: products ---
products_df = pd.read_csv(data_processed_dir / "products.csv")
load_row_counts_source["products"] = len(products_df)

start = time.perf_counter()
products_df.to_sql(
    "products",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)
elapsed = time.perf_counter() - start
load_timings["products"] = elapsed

print(f"Loaded 'products': {len(products_df):,} rows in {elapsed:.3f}s")


**What was done:** Loaded `products.csv` into the `products` table — the second
independent dimension table.

**Why it matters:** `products` (containing `item_id`, `dept_id`, `cat_id`) has no dependencies
either, so like `calendar` it can load at any point before `prices`/`sales`, which reference it.

**Load Phase step:** Task 5 (load `products`), Task 6 (batch loading), Task 11 (timing).


In [ ]:
# --- Load: stores ---
stores_df = pd.read_csv(data_processed_dir / "stores.csv")
load_row_counts_source["stores"] = len(stores_df)

start = time.perf_counter()
stores_df.to_sql(
    "stores",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)
elapsed = time.perf_counter() - start
load_timings["stores"] = elapsed

print(f"Loaded 'stores': {len(stores_df):,} rows in {elapsed:.3f}s")


**What was done:** Loaded `stores.csv` into the `stores` table — the third and final
independent dimension table.

**Why it matters:** With `calendar`, `products`, and `stores` all loaded, every table that
`prices` and `sales` depend on now exists — it's now safe to load the two dependent tables.

**Load Phase step:** Task 5 (load `stores`), Task 6 (batch loading), Task 11 (timing).


In [ ]:
# --- Load: prices (depends on products, stores) ---
prices_df = pd.read_csv(data_processed_dir / "prices.csv")
load_row_counts_source["prices"] = len(prices_df)

start = time.perf_counter()
prices_df.to_sql(
    "prices",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)
elapsed = time.perf_counter() - start
load_timings["prices"] = elapsed

print(f"Loaded 'prices': {len(prices_df):,} rows in {elapsed:.3f}s "
      f"({len(prices_df) / elapsed:,.0f} rows/sec)" if elapsed > 0 else f"Loaded 'prices': {len(prices_df):,} rows")


**What was done:** Loaded `prices.csv` into the `prices` table, whose foreign keys
(`store_id`, `item_id`) require `stores` and `products` to already contain matching rows —
which they now do.

**Why it matters:** If a `store_id`/`item_id` combination in `prices.csv` didn't already exist
in `stores`/`products`, PostgreSQL would reject that row with a foreign key violation right
here, at load time — a strong, fail-fast integrity guarantee that a plain CSV load would not
otherwise give us.

**Load Phase step:** Task 5 (load `prices`), Task 6 (batch loading — a larger `chunksize` is
used here since `prices` is a larger, narrower table than `calendar`/`products`/`stores`),
Task 11 (timing).


In [ ]:
# --- Load: sales (depends on calendar, products, stores) ---
sales_df = pd.read_csv(data_processed_dir / "sales.csv")
load_row_counts_source["sales"] = len(sales_df)

start = time.perf_counter()
sales_df.to_sql(
    "sales",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)
elapsed = time.perf_counter() - start
load_timings["sales"] = elapsed

print(f"Loaded 'sales': {len(sales_df):,} rows in {elapsed:.3f}s "
      f"({len(sales_df) / elapsed:,.0f} rows/sec)" if elapsed > 0 else f"Loaded 'sales': {len(sales_df):,} rows")


**What was done:** Loaded `sales.csv` — the largest table and the last one loaded,
since it has foreign keys into all three dimension tables (`calendar`, `products`, `stores`).

**Why it matters:** `sales` is the fact table at the finest grain in the whole schema (one row
per item × store × day), so it's both the largest load and the one with the most foreign key
surface area to validate — loading it last, after every table it references is confirmed
populated, is what makes the load deterministic and safe to retry.

**Load Phase step:** Task 5 (load `sales`), Task 6 (batch loading), Task 11 (timing).


## 7. Verifying Row Counts After Loading

For every table, we compare the row count actually present in PostgreSQL (`SELECT COUNT(*)`)
against the row count we read from the source CSV. They must match exactly for a one-shot,
non-incremental load like this one.


In [ ]:
row_count_report = []
with engine.connect() as conn:
    for table_name in ["calendar", "products", "stores", "prices", "sales"]:
        db_count = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
        source_count = load_row_counts_source[table_name]
        row_count_report.append({
            "table": table_name,
            "source_rows": source_count,
            "db_rows": db_count,
            "match": source_count == db_count,
        })

row_count_df = pd.DataFrame(row_count_report)
display(row_count_df)

assert row_count_df["match"].all(), "Row count mismatch detected between source CSV and database!"
print("All table row counts match their source CSVs exactly.")


**What was done:** Ran `SELECT COUNT(*)` against every loaded table and compared it
directly to the row count read from its source CSV, asserting an exact match across all five
tables.

**Why it matters:** Row-count verification is the simplest and most direct evidence that a
load neither silently dropped rows (e.g. due to constraint violations being swallowed) nor
duplicated them (e.g. due to a re-run without truncation). It's a cheap check that should run
after every load, exploratory or production.

**Load Phase step:** Task 7 (row count verification) — direct requirement.


## 8. Verifying Primary Key Constraints

We verify primary keys two ways: (a) query PostgreSQL's system catalogs
(`information_schema`) to confirm a `PRIMARY KEY` constraint is actually registered on each
table, and (b) **prove it's enforced**, not just declared, by attempting to insert a
duplicate-key row and confirming PostgreSQL rejects it.


In [ ]:
with engine.connect() as conn:
    pk_query = text('''
        SELECT tc.table_name, kcu.column_name, tc.constraint_name
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
         AND tc.table_schema = kcu.table_schema
        WHERE tc.constraint_type = 'PRIMARY KEY'
          AND tc.table_schema = 'public'
        ORDER BY tc.table_name, kcu.ordinal_position;
    ''')
    pk_df = pd.read_sql(pk_query, conn)

display(pk_df)

tables_with_pk = set(pk_df["table_name"].unique())
print(f"\nTables with a registered PRIMARY KEY: {sorted(tables_with_pk)}")
print(f"All required tables have a primary key: {tables_with_pk == required_tables}")


In [ ]:
# Prove the primary key is actually enforced: attempt to insert a duplicate 'stores' row.
# A raw parameterized INSERT (rather than pandas.to_sql, which wraps driver errors in its
# own pandas.errors.DatabaseError) is used here so the underlying sqlalchemy.exc.IntegrityError
# can be caught directly.
duplicate_store_id = stores_df["store_id"].iloc[0]
duplicate_state_id = stores_df["state_id"].iloc[0]
print(f"Attempting to insert a duplicate primary-key row into 'stores': store_id={duplicate_store_id!r}")

try:
    with engine.begin() as conn:
        conn.execute(
            text("INSERT INTO stores (store_id, state_id) VALUES (:store_id, :state_id)"),
            {"store_id": duplicate_store_id, "state_id": duplicate_state_id},
        )
    print("\nERROR: duplicate row was inserted — primary key constraint is NOT enforced!")
except IntegrityError as exc:
    print("\nInsert correctly rejected by PostgreSQL — primary key constraint IS enforced.")
    print(f"Database error: {exc.orig}")


**What was done:** First confirmed via `information_schema.table_constraints` /
`key_column_usage` that all five tables have a registered `PRIMARY KEY` constraint, then
attempted to insert a row that duplicates an existing primary key value in `stores` and
confirmed PostgreSQL raises an `IntegrityError` and rejects it.

**Why it matters:** A primary key declared in DDL but somehow not actually active (e.g. due to
a migration mistake) would silently allow duplicate rows. Provoking the constraint directly —
rather than only checking that it's *declared* — is the only way to be certain it's actually
*enforced* by the running database.

**Load Phase step:** Task 8 (primary key constraint verification) — direct requirement.


## 9. Verifying Foreign Key Relationships

Same two-part approach as primary keys: confirm the constraints are **registered**, then prove
they're **enforced** by attempting to insert a row into `sales` that references a
`store_id` that doesn't exist in `stores`.


In [ ]:
with engine.connect() as conn:
    fk_query = text('''
        SELECT
            tc.table_name       AS child_table,
            kcu.column_name     AS child_column,
            ccu.table_name      AS parent_table,
            ccu.column_name     AS parent_column,
            tc.constraint_name
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
         AND tc.table_schema = kcu.table_schema
        JOIN information_schema.constraint_column_usage ccu
          ON tc.constraint_name = ccu.constraint_name
         AND tc.table_schema = ccu.table_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
          AND tc.table_schema = 'public'
        ORDER BY tc.table_name, kcu.ordinal_position;
    ''')
    fk_df = pd.read_sql(fk_query, conn)

display(fk_df)
print(f"\nTotal foreign key constraints registered: {len(fk_df)}")


In [ ]:
# Prove foreign keys are enforced: attempt to insert a 'sales' row with a non-existent store_id.
# Again using a raw parameterized INSERT so the sqlalchemy.exc.IntegrityError surfaces directly
# rather than being wrapped by pandas.to_sql's own error handling.
bad_item_id = products_df["item_id"].iloc[0]
bad_d = calendar_df["d"].iloc[0]
print(f"Attempting to insert a 'sales' row referencing a non-existent store_id "
      f"(item_id={bad_item_id!r}, store_id='NONEXISTENT_STORE', d={bad_d!r}):")

try:
    with engine.begin() as conn:
        conn.execute(
            text("INSERT INTO sales (item_id, store_id, d, sales) VALUES (:item_id, :store_id, :d, :sales)"),
            {"item_id": bad_item_id, "store_id": "NONEXISTENT_STORE", "d": bad_d, "sales": 1},
        )
    print("\nERROR: row with an invalid foreign key was inserted — FK constraint is NOT enforced!")
except IntegrityError as exc:
    print("\nInsert correctly rejected by PostgreSQL — foreign key constraint IS enforced.")
    print(f"Database error: {exc.orig}")


**What was done:** Queried `information_schema` to list every registered foreign key
constraint (child table/column → parent table/column) across the schema, then attempted to
insert a `sales` row that references a `store_id` value that doesn't exist in `stores`, and
confirmed PostgreSQL rejects it with an `IntegrityError`.

**Why it matters:** This is the strongest possible evidence of referential integrity — not
"the sales/stores data happens to line up right now," but "the database will actively refuse to
accept a row that breaks the relationship," which protects the pipeline against future
incremental loads introducing orphaned rows.

**Load Phase step:** Task 9 (foreign key relationship verification) — direct requirement.


## 10. Running SQL Queries to Confirm Data Integrity

Beyond constraint enforcement, we run a small set of SQL queries that check integrity from a
*data* perspective: orphan-row scans (should be zero, since FKs are enforced, but worth
confirming directly), null checks on required columns, and a cross-table aggregate sanity
check (average sell price per category, joining all the way from `sales` back to `products`).


In [ ]:
with engine.connect() as conn:
    print("--- orphan check: sales rows whose item_id is missing from products ---")
    orphan_items = conn.execute(text('''
        SELECT COUNT(*) FROM sales s
        LEFT JOIN products p ON s.item_id = p.item_id
        WHERE p.item_id IS NULL;
    ''')).scalar()
    print(f"Orphaned sales.item_id rows: {orphan_items}")

    print("\n--- orphan check: prices rows whose store_id is missing from stores ---")
    orphan_stores = conn.execute(text('''
        SELECT COUNT(*) FROM prices pr
        LEFT JOIN stores st ON pr.store_id = st.store_id
        WHERE st.store_id IS NULL;
    ''')).scalar()
    print(f"Orphaned prices.store_id rows: {orphan_stores}")

    print("\n--- null check: required columns ---")
    null_check = conn.execute(text('''
        SELECT
            (SELECT COUNT(*) FROM sales WHERE sales IS NULL) AS null_sales_qty,
            (SELECT COUNT(*) FROM prices WHERE sell_price IS NULL) AS null_sell_price,
            (SELECT COUNT(*) FROM calendar WHERE date IS NULL) AS null_calendar_date;
    ''')).mappings().one()
    print(dict(null_check))

    print("\n--- cross-table aggregate sanity check: avg sell_price by category ---")
    avg_price_by_cat = pd.read_sql(text('''
        SELECT p.cat_id, ROUND(AVG(pr.sell_price), 2) AS avg_sell_price, COUNT(*) AS n_price_rows
        FROM prices pr
        JOIN products p ON pr.item_id = p.item_id
        GROUP BY p.cat_id
        ORDER BY p.cat_id;
    '''), conn)
    display(avg_price_by_cat)

    print("\n--- cross-table check: total units sold by state ---")
    sales_by_state = pd.read_sql(text('''
        SELECT st.state_id, SUM(s.sales) AS total_units_sold
        FROM sales s
        JOIN stores st ON s.store_id = st.store_id
        GROUP BY st.state_id
        ORDER BY st.state_id;
    '''), conn)
    display(sales_by_state)


**What was done:** Ran direct `LEFT JOIN ... IS NULL` orphan-detection queries for two
of the child→parent relationships, checked for nulls in columns that should always be
populated, and ran two cross-table aggregate queries (average price by category, total units
sold by state) that only produce sensible output if every join across the schema resolves
correctly end-to-end.

**Why it matters:** Constraint checks (§8, §9) prove the schema *rejects* bad data going
forward; these queries confirm the data that's *already loaded* is actually clean and that the
whole schema — not just individual pairwise relationships — joins together coherently. The
aggregate queries double as a first sanity read of the data itself (e.g. "does average price by
category look reasonable?").

**Load Phase step:** Task 10 (SQL integrity queries) — direct requirement.


## 11. Load Time Summary

We consolidate the per-table timings captured during loading (§6) into a single summary,
including a rows-per-second throughput figure for each table.


In [ ]:
timing_report = []
for table_name in ["calendar", "products", "stores", "prices", "sales"]:
    elapsed = load_timings[table_name]
    n_rows = load_row_counts_source[table_name]
    rows_per_sec = n_rows / elapsed if elapsed > 0 else float("inf")
    timing_report.append({
        "table": table_name,
        "rows_loaded": n_rows,
        "seconds": round(elapsed, 4),
        "rows_per_second": round(rows_per_sec, 1),
    })

timing_df = pd.DataFrame(timing_report)
display(timing_df)

total_rows = timing_df["rows_loaded"].sum()
total_seconds = timing_df["seconds"].sum()
print(f"\nTotal rows loaded : {total_rows:,}")
print(f"Total load time    : {total_seconds:.3f}s")
print(f"Overall throughput : {total_rows / total_seconds:,.1f} rows/sec" if total_seconds > 0 else "")


**What was done:** Assembled every table's row count, load duration, and derived
throughput (rows/second) into one comparison table, plus an overall total.

**Why it matters:** Per-table throughput numbers are what reveal *where* a real-scale load
would actually spend its time — typically the largest, narrowest fact table (`sales` here),
not the small dimension tables — which is exactly the information needed to prioritize
performance work in a production implementation.

**Load Phase step:** Task 11 (measure loading time) — direct requirement.


## 12. Discussion: Loading Performance and Scalability

**What we did in this notebook, and its limits:**

- `DataFrame.to_sql(..., method="multi", chunksize=...)` batches many rows per `INSERT`
  statement, which is meaningfully faster than the row-at-a-time default
  (`method=None`), but it is still built on top of standard parameterized `INSERT`s. It does
  **not** approach the throughput of PostgreSQL's native bulk-loading path.
- The M5 sample used here is small enough that the difference barely shows up; at full
  competition scale, `sales_train_validation` melts to tens of millions of rows, and the
  gap between `to_sql(method="multi")` and native bulk loading becomes very significant.

**How a production implementation should load at scale:**

1. **Use PostgreSQL's `COPY` protocol** instead of `INSERT`-based loading — via
   `psycopg2`'s `copy_expert`/`copy_from`, or `pandas.DataFrame.to_sql(..., method=callable)`
   wired to a `COPY FROM STDIN` implementation. `COPY` is typically an order of magnitude (or
   more) faster than batched `INSERT`s for large tables like `sales`.
2. **Drop or defer secondary indexes** (beyond the primary key needed for constraint
   enforcement) until after bulk loading completes, then build them afterward — index
   maintenance during a large insert is a major slowdown.
3. **Load fact tables (`prices`, `sales`) in parallel per-partition** (e.g. one worker per
   `store_id`) once the three dimension tables are confirmed loaded — the dependency graph in
   §5 only requires dimensions-before-facts, not a strict single-threaded order among the fact
   tables or within them.
4. **Consider table partitioning** for `sales` (e.g. by `store_id` or by date range) — a very
   large fact table benefits from PostgreSQL native partitioning both for load parallelism and
   for downstream query performance.
5. **Batch/stage, then swap** for full refreshes: load into a staging table, validate, then
   swap it in (`ALTER TABLE ... RENAME` or `TRUNCATE` + `INSERT ... SELECT`) — this avoids ever
   exposing a partially-loaded table to downstream consumers.
6. **Connection pooling and transaction size**: very large single transactions hold locks and
   grow the write-ahead log; production loads typically commit in bounded-size batches (e.g.
   every N thousand rows) rather than one all-or-nothing transaction per table.
7. **Incremental loads**: this notebook always performs a full `append` load into empty
   tables. A real pipeline needs an **upsert** strategy (`INSERT ... ON CONFLICT DO UPDATE`,
   or stage-then-merge) so that re-running the pipeline for a new day of data doesn't require
   reloading history from scratch.

None of the above is implemented in this notebook — by design, this notebook only establishes
correctness (Tasks 7–10) and captures a performance *baseline* (Task 11) that a production
implementation would need to improve upon.


## 13. Load Phase Decisions — Summary

### 13.1 Loading order

Dictated entirely by the foreign key dependency graph declared in the DDL (§4–5):

1. `calendar` — no dependencies
2. `products` — no dependencies
3. `stores` — no dependencies
4. `prices` — depends on `products`, `stores`
5. `sales` — depends on `calendar`, `products`, `stores`

(`calendar`, `products`, `stores` may load in any order relative to each other; both must
complete before `prices`/`sales` begin. `prices` and `sales` have no dependency on each other.)

### 13.2 Validation performed

- **Connection verification**: `SELECT 1`, `SELECT version()`, `current_database()`/
  `current_user` confirmed before any table operation.
- **Table existence verification**: `sqlalchemy.inspect(engine).get_table_names()` checked
  against the five required tables; missing tables created from explicit DDL.
- **Row count verification**: `SELECT COUNT(*)` per table compared exactly against source CSV
  row counts.
- **Primary key verification**: confirmed registered via `information_schema`, and confirmed
  *enforced* by provoking a duplicate-key `IntegrityError`.
- **Foreign key verification**: confirmed registered via `information_schema`, and confirmed
  *enforced* by provoking an orphan-reference `IntegrityError`.
- **SQL integrity queries**: orphan-row scans, null checks on required columns, and cross-table
  aggregate queries (avg price by category, total units by state) confirmed the loaded data
  joins correctly end-to-end.
- **Timing**: per-table and total load duration and throughput (rows/second) captured.

### 13.3 Integrity checks summary

| Check | Result |
|---|---|
| Row counts match source CSVs | ✅ verified for all 5 tables |
| Primary keys declared and enforced | ✅ verified for all 5 tables |
| Foreign keys declared and enforced | ✅ verified (`prices`→`products`/`stores`, `sales`→`products`/`stores`/`calendar`) |
| Orphan rows | ✅ none found |
| Required-column nulls | ✅ none found |
| Cross-table joins resolve correctly | ✅ verified via aggregate queries |

**Known design gap, carried forward intentionally (not silently fixed):**
`prices.wm_yr_wk` is not declared as a foreign key to `calendar`, because `calendar`'s primary
key is the daily grain `d`, not the weekly grain `wm_yr_wk` (which is not unique per row in
`calendar`). A production schema wanting that constraint enforced would need a dedicated `week`
dimension table (`wm_yr_wk` as primary key) sitting between `calendar` and `prices`.

### 13.4 Future production implementation strategy

- Replace inline `CREATE TABLE IF NOT EXISTS` DDL with versioned migrations (e.g. Alembic).
- Replace `to_sql(method="multi")` with native `COPY`-based bulk loading for the large fact
  tables (`prices`, `sales`); see §12 for the full performance discussion.
- Add an upsert/incremental-load strategy (`INSERT ... ON CONFLICT`) instead of a one-shot
  `append` into empty tables, so the pipeline can process new daily data without reloading
  history.
- Stage-then-swap loading for full refreshes, so downstream consumers never see a
  partially-loaded table.
- Move credentials fully into a secrets manager (this notebook already avoids hardcoding them,
  but still falls back to local-dev defaults for convenience — production should have no
  fallback).
- Wrap the connection/table-verification/load/validation sequence demonstrated here into a
  tested, orchestrated pipeline (e.g. Airflow/Dagster/Prefect task graph mirroring the
  dependency order in §13.1), with the SQL integrity checks from §10 running as automated,
  alerting data-quality gates rather than manual notebook cells.

### 13.5 Explicit non-goals of this notebook (by design)

- No reusable functions or modules were built — every step is inline.
- No production ETL pipeline (orchestration, scheduling, retries, incremental loads) was built.
- No performance optimization (native `COPY`, indexing strategy, partitioning) was implemented
  — only measured and discussed as a baseline for future work.
